In [1]:
import os
import shutil
import subprocess

# --- CONFIGURATION ---
INPUT_DIR = "/kaggle/input/datasets/phongviet/2dgs-code" 
WORKING_DIR = "/kaggle/working/MonoGS"

# 1. Install Dependencies with Exact Versions from your Local Env
# 1. Install closest possible versions for Python 3.12
# Note: numpy 1.26.4 is the 1.x series compatible with Python 3.12
!pip install \
    "numpy<2.0" \
    plyfile==0.8.1 \
    munch==4.0.0 \
    trimesh==4.11.3 \
    evo==1.11.0 \
    open3d==0.19.0 \
    torchmetrics==1.5.2 \
    imgviz==1.7.5 \
    PyOpenGL==3.1.10 \
    glfw==2.10.0 \
    PyGLM==2.8.3 \
    wandb==0.24.2 \
    lpips==0.1.4 \
    rich==14.3.3 \
    ruff==0.15.5 \
    ninja \
    -q
# 2. Copy Code to Writable Workspace
if not os.path.exists(WORKING_DIR):
    print(f"Copying code to {WORKING_DIR}...")
    shutil.copytree(INPUT_DIR, WORKING_DIR)
else:
    print("Code already exists in /working.")
%cd {WORKING_DIR}

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.0/114.0 kB 5.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 19.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.5/740.5 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 891.4/891.4 kB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.5/243.5 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
%cd {WORKING_DIR}

# --- FIX: simple_knn.cu requires <cfloat> ---
print("Applying simple-knn header fix...")
!sed -i '1i #include <cfloat>' submodules/simple-knn/simple_knn.cu

# 1. Install Gaussian Rasterization
print("Building diff-gaussian-rasterization...")
%cd {WORKING_DIR}/submodules/diff-gaussian-rasterization
!pip install . --no-build-isolation -q

# 2. Install Surfel Rasterization
print("Building diff-surfel-rasterization...")
%cd {WORKING_DIR}/submodules/diff-surfel-rasterization
!pip install . --no-build-isolation -q

# 3. Install Simple-KNN
print("Building simple-knn...")
%cd {WORKING_DIR}/submodules/simple-knn
!rm -rf build/ dist/ *.egg-info
!pip install . --no-build-isolation -q

%cd {WORKING_DIR}
print("\nEnvironment preparation complete!")

/kaggle/working/MonoGS
Applying simple-knn header fix...
Building diff-gaussian-rasterization...
/kaggle/working/MonoGS/submodules/diff-gaussian-rasterization
  Preparing metadata (setup.py) ... done
Building diff-surfel-rasterization...
/kaggle/working/MonoGS/submodules/diff-surfel-rasterization
  Preparing metadata (setup.py) ... done
Building simple-knn...
/kaggle/working/MonoGS/submodules/simple-knn
  Preparing metadata (setup.py) ... done
/kaggle/working/MonoGS

Environment preparation complete!


In [3]:
!mkdir -p datasets/tum
%cd datasets/tum

def download_tum_quiet(url, folder_name):
    if not os.path.exists(folder_name):
        print(f"Downloading {folder_name}...")
        tar_file = url.split("/")[-1]
        !wget -q {url}
        !tar -xzf {tar_file} > /dev/null
        !rm {tar_file}
        print(f"Done.")
    else:
        print(f"Skipping {folder_name} (already exists)")

datasets = [
    ("https://cvg.cit.tum.de/rgbd/dataset/freiburg1/rgbd_dataset_freiburg1_desk.tgz", "rgbd_dataset_freiburg1_desk"),
    ("https://cvg.cit.tum.de/rgbd/dataset/freiburg2/rgbd_dataset_freiburg2_xyz.tgz", "rgbd_dataset_freiburg2_xyz"),
    ("https://cvg.cit.tum.de/rgbd/dataset/freiburg3/rgbd_dataset_freiburg3_long_office_household.tgz", "rgbd_dataset_freiburg3_long_office_household")
]

for url, folder in datasets:
    download_tum_quiet(url, folder)

%cd {WORKING_DIR}

/kaggle/working/MonoGS/datasets/tum
Done.
Done.
Done.
/kaggle/working/MonoGS


In [4]:
configs = [
    "configs/mono/tum/fr1_desk.yaml",
    "configs/mono/tum/fr2_xyz.yaml",
    "configs/mono/tum/fr3_office.yaml"
]

for config_path in configs:
    name = config_path.split("/")[-1].replace(".yaml", "")
    print(f"\n{'#'*30}\n# RUNNING EVAL: {name}\n{'#'*30}")
    
    # Run evaluation in headless mode
    try:
        subprocess.run(["python", "slam.py", "--config", config_path, "--eval"], check=True)
    except subprocess.CalledProcessError as e:
        print(f"Error on {name}: {e}")

print("\nAll evaluations finished. Check the /results/ folder for outputs.")

# Bonus: Create a ZIP of all results for easy download
!zip -r monogs_results.zip results/ > /dev/null


##############################
# RUNNING EVAL: fr1_desk
##############################
Initialized new /root/.evo/settings.json
MonoGS: Running MonoGS in Evaluation Mode
MonoGS: Following config will be overriden
MonoGS:         save_results=True
MonoGS:         use_gui=False
MonoGS:         eval_rendering=True
MonoGS:         use_wandb=False
MonoGS: saving results in 
results/tum_rgbd_dataset_freiburg1_desk/2026-04-05-05-54-16
MonoGS: Resetting the system
MonoGS: Initialized map
MonoGS: Resetting the opacity of non-visible Gaussians
MonoGS: Performing initial BA for initialization
MonoGS: Initialized SLAM
MonoGS: Evaluating ATE at frame:  51
Eval: RMSE ATE  (6608 gaussians) 0.022124870965303498
MonoGS: Evaluating ATE at frame:  104
Eval: RMSE ATE  (13175 gaussians) 0.02893098895768308
MonoGS: Evaluating ATE at frame:  149
Eval: RMSE ATE  (16925 gaussians) 0.032203533207159935
MonoGS: Evaluating ATE at frame:  186
Eval: RMSE ATE  (24139 gaussians) 0.04038420635087293
MonoGS: Resetting

100%|██████████| 233M/233M [00:01<00:00, 194MB/s]


Eval: mean psnr: 17.954130971154502, ssim: 0.6694763316664585, lpips: 
0.3754862023647441
MonoGS: Backend stopped and joined the main thread
MonoGS: Done.

##############################
# RUNNING EVAL: fr2_xyz
##############################
MonoGS: Running MonoGS in Evaluation Mode
MonoGS: Following config will be overriden
MonoGS:         save_results=True
MonoGS:         use_gui=False
MonoGS:         eval_rendering=True
MonoGS:         use_wandb=False
MonoGS: saving results in 
results/tum_rgbd_dataset_freiburg2_xyz/2026-04-05-06-27-46
MonoGS: Resetting the system
MonoGS: Initialized map
MonoGS: Resetting the opacity of non-visible Gaussians
MonoGS: Performing initial BA for initialization
MonoGS: Initialized SLAM
MonoGS: Evaluating ATE at frame:  102
Eval: RMSE ATE  (8187 gaussians) 0.004410389354361265
MonoGS: Resetting the opacity of non-visible Gaussians
MonoGS: Evaluating ATE at frame:  371
Eval: RMSE ATE  (16092 gaussians) 0.007756977934256538
MonoGS: Evaluating ATE at frame: 